In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

from study.data import cohort, features, prediction_metadata
from study.io import WORK, load, reset_directory, write_table
from study.metrics import binary_losses
from study.models import fit_model, fit_source_model, predict
from study.statistics import paired_difference, prediction_rows, risk_matrix

In [ ]:
reset_directory(WORK / 'predictions/learners')

source_fits = load(WORK / 'models/source.pkl')
shared = load(WORK / 'feature_sets.pkl')['shared']
x, metadata = features('metavision'), cohort('metavision')

SETTINGS = {
    'xgboost_depth2': ('xgboost', dict(source_fits['parameters'], max_depth = 2)),
    'histogram': (
        'histogram',
        {
            'max_iter': 100,
            'max_leaf_nodes': 7,
            'learning_rate': 0.05,
            'l2_regularization': 3,
            'early_stopping': False,
            'random_state': 77
        }
    ),
    'logistic': ('logistic', {'C': 0.01, 'solver': 'lbfgs', 'max_iter': 3000, 'random_state': 77}),
    'random_forest': (
        'random_forest',
        {
            'n_estimators': 300,
            'criterion': 'log_loss',
            'max_features': 'sqrt',
            'min_samples_leaf': 20,
            'bootstrap': True,
            'n_jobs': 1,
            'random_state': 77
        }
    )
}

In [ ]:
# Fit fixed alternative learners
rows, diagnostics = [], []
for repeat in source_fits['repeats']:
    fit_ids = source_fits['fit_ids']['metavision', repeat]

    # Fit each fixed learner across the same feature tiers
    rows = []
    for name, (family, parameters) in SETTINGS.items():
        for tier in ('R1', 'R2', 'R3'):
            with threadpool_limits(limits = 1):
                fitted = fit_source_model(
                    x.loc[fit_ids, shared[tier]], metadata.loc[fit_ids, 'in_hospital_mortality'], family, parameters
                )
                model = fitted['model']

            # Check logistic regression convergence and record numerical diagnostics
            if family == 'logistic':
                matrix = model['transform'].transform(x.loc[fit_ids, shared[tier]])
                estimator = model['estimator']
                linear = np.einsum('ij,j->i', matrix, estimator.coef_[0]) + estimator.intercept_[0]
                residual = expit(linear) - metadata.loc[fit_ids, 'in_hospital_mortality'].to_numpy()
                gradient = (np.einsum('ij,i->j', matrix, residual) + estimator.coef_[0] / 0.01) / len(fit_ids)

                maximum_gradient = max(np.max(np.abs(gradient)), abs(residual.mean()))

                diagnostics.append(
                    dict(
                        tier = tier,
                        repeat = repeat,
                        finite_matrix = np.isfinite(matrix).all(),
                        finite_parameters = np.isfinite(estimator.coef_).all(),
                        maximum_gradient = maximum_gradient,
                    )
                )

            # Use source OOF predictions and frozen model target predictions
            for domain in ('metavision', 'mimiciv', 'eicu'):
                target = cohort(domain).sort_index()
                ids = (
                    target.index[target.partition.eq('test')]
                    if domain == 'eicu'
                    else target.index
                )

                risk = (
                    fitted['oof'].loc[ids].to_numpy()
                    if domain == 'metavision'
                    else predict(model, features(domain).loc[ids])
                )

                rows.append(
                    prediction_rows(
                        prediction_metadata(target.loc[ids]),
                        risk,
                        learner = name,
                        repeat = repeat,
                        tier = tier,
                        domain = domain,
                        calibration = 'raw',
                    )
                )

    write_table(WORK / 'predictions/learners' / f'{repeat}.parquet', pd.concat(rows, ignore_index = True))
    print('Alternative learners allocation', repeat)

predictions = pd.read_parquet(WORK / 'predictions/learners')
write_table(WORK / 'results/logistic_numerics.parquet', pd.DataFrame(diagnostics))

In [ ]:
# Compare information increments within each learner
rows = []
for (learner, domain, calibration), group in predictions.groupby(['learner', 'domain', 'calibration']):
    ids = pd.Index(sorted(group.stay_id.unique()))
    metadata = prediction_metadata(cohort(domain).loc[ids])

    # Compare adjacent tiers on the same patients
    for richer, simpler in [('R2', 'R1'), ('R3', 'R2')]:
        for equal_hospital in [False, True] if domain == 'eicu' else [False]:
            result = paired_difference(
                metadata,
                risk_matrix(group.loc[group.tier.eq(richer)], ids),
                risk_matrix(group.loc[group.tier.eq(simpler)], ids),
                cluster = domain == 'eicu',
                equal_hospital = equal_hospital,
            )

            rows.append(
                dict(
                    learner = learner,
                    domain = domain,
                    calibration = calibration,
                    contrast = richer + '_minus_' + simpler,
                    **result,
                )
            )

results = pd.DataFrame(rows)

# Adjust p values within each target and weighting family
for _, family in results.groupby(['domain', 'weighting']):
    results.loc[family.index, 'holm_p'] = multipletests(family.p_value, method = 'holm')[1]

write_table(WORK / 'results/learners.parquet', results)

In [ ]:
# Fit R1–R3 on five 80% source subsamples
metadata_source = cohort('metavision')
target = cohort('eicu').query("partition == 'test'").sort_index()
rows = []

for repeat in source_fits['repeats']:
    fit_ids = source_fits['fit_ids']['metavision', repeat]

    # Sample source patients without replacement
    for seed in range(5):
        sample = np.random.default_rng(seed).choice(fit_ids, int(0.8 * len(fit_ids)), replace = False)
        sample = pd.Index(sorted(sample))
        probabilities = {}

        # Fit all tiers on the same source subsample
        for tier in ('R1', 'R2', 'R3'):
            model = fit_model(
                x.loc[sample, shared[tier]],
                metadata_source.loc[sample, 'in_hospital_mortality'],
                'xgboost',
                source_fits['parameters'],
            )
            probabilities[tier] = predict(model, features('eicu').loc[target.index])

        # Compare paired target losses for adjacent tiers
        for richer, simpler in [('R2', 'R1'), ('R3', 'R2')]:
            delta = binary_losses(
                target.in_hospital_mortality, probabilities[richer], epsilon = 1e-08
            ) - binary_losses(target.in_hospital_mortality, probabilities[simpler], epsilon = 1e-08)

            rows.append(
                dict(seed = seed, repeat = repeat, contrast = richer + '_minus_' + simpler, difference = delta.mean())
            )

per_allocation = pd.DataFrame(rows)
write_table(WORK / 'results/source_subsamples_by_allocation.parquet', per_allocation)
averaged = per_allocation.groupby(['seed', 'contrast'])[['difference']].mean().reset_index()

write_table(WORK / 'results/source_subsamples.parquet', averaged)